###**SQL Generator using OSS Model - Defog sql coder -7b -2**

##1.Installation of packages

In [ ]:
!pip install torch transformers bitsandbytes accelerate sqlparse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

###Import Libraries

In [ ]:
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM

In [ ]:
torch.cuda.is_available()

True

In [ ]:
available_memory=torch.cuda.get_device_properties(0).total_memory
print(available_memory)

15828320256


##2. Download the SQL Model

In [ ]:
model_name="defog/sqlcoder-7b-2"
tokenizer=AutoTokenizer.from_pretrained(model_name)

#Downloading the model depends upon available memory
if available_memory>=16:
    model=AutoModelForCausalLM.from_pretrained(model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    use_cache=True,
    )

else:
    model=AutoModelForCausalLM.from_pretrained(model_name,
    trust_remote_code=True,
    load_in_8bit=True,
    device_map="auto",
    use_cache=True,
    )


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/3.59G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

##3. Set the Prompt and Database

In [ ]:
prompt="""
    ### TASK
    Given Database schema, generate SQL query for the user's request
    [QUESTION]{question}[/QUESTION]

    ### INSTRUCTIONS
    If the cannot answer the question with the given database,return "I DON'T KNOW"
    --REVENUE is the  price is multiplied by its quantity
    --COST is the supply_price is multiplied by its quantity

    ### DATABASE SCHEMA
    This is the database for answer the user's query.

    CREATE TABLE sales(
    sale_id INTEGER PRIMARY KEY,-- Unique id for each sale
    customer_id INTEGER,--Unique id for each CUSTOMER
    product_id INTEGER,--Unique id for each PRODUCT
    sale_date DATE,--Date of each sale
    quantity INTEGER,--Quantity of each product
    );

    CREATE TABLE products(
    product_id INTEGER PRIMARY KEY,--Unique id for each product
    product_name VARCHAR(50),--Name of each product
    quantity INTEGER,--Quantity in each stock of product
    unit_price DECIMAL(10,2),--Price of each product
    );

    CREATE TABLE customers(
    customer_id INTEGER PRIMARY KEY,--Unique id for each customer
    name VARCHAR(50),--Name of customer
    phone_no INTEGER,--Phone number of customer
    address VARCHAR(100),--Address of customer
    );


    --sales.product_id can be joined with products.product_id
    --sales.customer_id can be joined with customers.customer_id


    ### ANSWER
    Given Database schema, answer the user's request and generate the SQL.
    [QUESTION]{question}[/QUESTION]
    [SQL]
    """

##4.Generate the SQL

In [ ]:
import sqlparse
def generate_query(question):
    updated_prompt=prompt.format(question=question)
    inputs=tokenizer(updated_prompt,return_tensors="pt").to("cuda")
    generated_ids=model.generate(
       **inputs,
       num_return_sequences=1,
       eos_token_id=tokenizer.eos_token_id,
       pad_token_id=tokenizer.eos_token_id,
       max_new_tokens=500,
       do_sample=False,
       num_beams=1,
    )

    outputs=tokenizer.batch_decode(generated_ids,skip_special_tokens=True)

    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    return sqlparse.format(outputs[0].split("[SQL]")[-1],reindent=True)


In [ ]:
question="What was the Highest quantity sold last month?"
print(generate_query(question))


SELECT MAX(s.quantity) AS max_quantity
FROM sales s
WHERE s.sale_date >= (CURRENT_DATE - INTERVAL '1 month');


In [ ]:
question="When did maximum amount of sales happened?"
print(generate_query(question))


SELECT MAX(s.sale_date) AS latest_sale_date
FROM sales s;
